# Modelling the supply of the Estionian energy

To model a the complex energy supply for estonian network we used a Spatio-Temporal Graph Neural Network (ST-GNN). It captures 7 years of grid data (2019-2025) from Elering API and weather data from Open-Meteo. The Elering API data provided us with features as production, renewable production, prices, consumption for Estonia, and flows between Estonia and respectively Finland, Latvia, Lithauen and Russian connections untill they were cut in ... 2025. Our purpose was to model the supply fpr January 2026. Therefore, we used Estonia, Findland, Latvia and Lithauen as nodes in our GNN as we had best data. 

For the most clear model, we used production in Estonia as our target variable. We therefore used lagged production as input, together with available energy, defined as production + flows from the node countries, renewable energy,  cyclic calender features, and weather data (temperature and wind speed), and energy prices. We also use the actual wind production from entosoe.

The node features included energy prices and flows to estonia. Negative flow imply imporrt and positive flow imply export. 

# Detailed Description of Supply.py

## Overview

**Supply.py** is the main training and evaluation pipeline for an **Spatio-Temporal Graph Neural Network (ST-GNN)** that predicts Estonian electricity available energy (production + imports - exports) under various scenarios. The script:

1. Fetches 7 years of grid data (2019–2026) from Elering API and weather from Open-Meteo
2. Engineers 14 features capturing grid dynamics, weather, and calendar patterns
3. Creates sequences for supervised learning with explicit **lag24 feature engineering** to prevent target leakage
4. Trains a quantile-regression GNN to predict P10, P50, P90 energy supply forecasts
5. Evaluates on unseen January 2026 data (the actual energy crisis)
6. Runs counterfactual scenarios (full isolation, isolation + wind investment) with realistic wind production from weather-based modeling

## 1. Data Collection & Preprocessing (Lines 15–76)

### 1.1 API Fetching
```
START = "2019-01-01T00:00:00.000Z"
END   = "2026-02-01T00:00:00.000Z"
```

- **Prices**: NPS prices from Elering for EE, FI, LV, LT (15-min granularity)
- **Cross-border flows**: Power flows EE↔FI, EE↔LV, EE↔RU_Narva, EE↔RU_Pihkva (hourly)
- **System production**: Total EE production + renewable + frequency deviation (5-min to hourly)
- **Weather**: Temperature, wind speed at 10m from Open-Meteo for central Estonia (58.90°N, 24.75°E)

### 1.2 Resampling to Hourly
All data standardized to hourly frequency via `.resample("h").mean()`. This handles:
- 15-min price data → hourly average
- 5-min production data → hourly average
- Already-hourly flows → passed through for alignment

### 1.3 Handling Missing Data
```python
df_daily[flow_cols] = df_daily[flow_cols].fillna(0)  # Flows default to 0 (no trade)
df_daily = df_daily.dropna(how="all")  # Drop rows that are entirely missing
```

## 2. Feature Engineering & Energy Balance (Lines 82–189)

### 2.1 Available Energy (Target Definition)
```python
system_h["available_energy"] = (
    system_h["production"]
    - flows_h[("ee", "fi")]
    - flows_h[("ee", "lv")] 
    - flows_h[("ee", "ru_narva")]
    - flows_h[("ee", "ru_pihkva")]
)
```

**Key insight**: Elering API convention is:
- **Positive flow** = Estonia exports (reduces available energy)
- **Negative flow** = Estonia imports (increases available energy)

Available energy = domestic production − exports + imports = production − (all flows, with signs preserved)

Diagnostic checks (lines 113–119) verify the assumption that EE is a net importer from FI and LV.

### 2.2 Calendar Features (Cyclical Encoding)
```python
hour_sin = np.sin(2π * idx.hour / 24)      # Hour-of-day (0–23)
dow_sin = np.sin(2π * idx.dayofweek / 7)   # Day-of-week (0–6)
month_sin = np.sin(2π * (idx.month−1) / 12) # Month-of-year (0–11)
```

Sine/cosine encoding prevents treating Mon=1, Sun=7 as numeric distance. Captures diurnal and seasonal patterns.

### 2.3 Grid Stress Signal
```python
freq_deviation = system_h["frequency"] - 50.0
```

Frequency drops below 50 Hz when supply is tight (high demand or low production). Real-time stress indicator.

### 2.4 Wind Feature
```python
wind_mw = entsoe["wind_onshore"].reindex(idx, method="ffill").fillna(0)
```

ENTSOE actual wind production data. Already included in `production`, but tracked separately so the model can learn wind-specific patterns and inject counterfactual scenarios.

### 2.5 Feature List (14 total per node)
**EE node (full data)**:
- `available_energy_lag24` ← **Lagged by HORIZON (24h) to prevent target leakage**
- `production_renewable`, `production`, `wind_mw` (energy sources)
- `flow_fi`, `flow_lv` (cross-border trade)
- `price` (market signal)
- `temperature`, `wind_speed_10m` (weather)
- `freq_deviation` (grid stress)
- `hour_sin`, `hour_cos`, `dow_sin`, `dow_cos`, `month_sin`, `month_cos` (calendar)

**FI/LV nodes (partial data, zero-padded for missing features)**:
- `price`, `flow_*` (neighboring countries' market + trade)
- All other features = 0

**LT node**:
- `price` only
- All other features = 0

### 2.6 Lag24 Leakage Prevention (Critical Design)
```python
"available_energy_lag24": system_h["available_energy"].shift(HORIZON)
```

**Problem**: If the model saw current available energy, it could "copy" recent values that overlap with the target window (24h ahead). Circular logic.

**Solution**: Shift available energy by 24h. The model only sees energy balance from before the prediction horizon:
- Input: available energy from hours [t−48, t−24)
- Target: available energy at hour t+23

No overlap → no leakage, but genuine predictive signal is preserved.

## 3. Graph Structure (Lines 241–245)

```python
edge_index = torch.tensor([
    [0, 1, 0, 2, 2, 3],  # source nodes
    [1, 0, 2, 0, 3, 2],  # target nodes (bidirectional)
], dtype=torch.long)
```

**Topology**: EE (node 0) at center:
- EE ↔ FI (nodes 0–1, bidirectional)
- EE ↔ LV (nodes 0–2, bidirectional)
- LV ↔ LT (nodes 2–3, bidirectional)

Models actual interconnections. Russia intentionally excluded (poor feature coverage).

## 4. Node Data Stacking (Lines 169–174)

```python
node_data = np.stack([
    ee_feats.values,   # node 0: EE (14 features)
    fi_feats.values,   # node 1: FI (14 features, mostly zeros)
    lv_feats.values,   # node 2: LV (14 features, mostly zeros)
    lt_feats.values,   # node 3: LT (14 features, mostly zeros)
], axis=1)
```

Shape: **(T, NUM_NODES, NUM_FEATURES)** = (60264 hours, 4 nodes, 14 features)

Ready for sequence creation.

## 5. Sequence Creation & Temporal Split (Lines 196–224)

### 5.1 Sliding Window Sequences
```python
SEQ_LEN = 48   # 2 days of history
HORIZON = 24   # predict 24 hours ahead

for t in range(SEQ_LEN, len(node_data) - HORIZON):
    X_list.append(node_data[t - SEQ_LEN:t])         # Input: [t-48, t)
    y_list.append(y_target_values[t + HORIZON - 1]) # Target: t+23
```

Each sample:
- **Input X**: 48 timesteps of (4 nodes, 14 features)
- **Target y**: scalar (available energy at t+23)

Total: ~60,000 sequences

### 5.2 Temporal Train/Val/Test Split
```python
jan2026_start = np.searchsorted(idx[SEQ_LEN:-HORIZON], pd.Timestamp("2026-01-01"))

X_train_full, y_train_full = X[:jan2026_start], y[:jan2026_start]
X_test, y_test = X[jan2026_start:], y[jan2026_start:]

val_split = int(len(X_train_full) * 0.8)
X_train, y_train = X_train_full[:val_split], y_train_full[:val_split]
X_val, y_val = X_train_full[val_split:], y_train_full[val_split:]
```

**Why temporal split?** Real-world forecasting: train on past, test on future. No information leakage.

- **Train**: 2019–Oct 2025 (~56k sequences)
- **Val**: Oct 2025–Dec 2025 (~5k sequences)
- **Test**: January 2026 (crisis month, ~744 sequences) — **held-out unseen data**

## 6. Normalization (Lines 227–238)

```python
scaler = ps.PowerScaler(X_train, y_train)  # Fit ONLY on training data

X_train_t = scaler.scale_x(X_train)
X_val_t = scaler.scale_x(X_val)
X_test_t = scaler.scale_x(X_test)

y_train_t = scaler.scale_y(y_train).view(-1, 1)
y_val_t = scaler.scale_y(y_val).view(-1, 1)
y_test_t = scaler.scale_y(y_test).view(-1, 1)
```

**PowerScaler** (custom normalization class):
- Fits mean and std on training data only
- Applies same transformation to val/test
- Prevents leakage: val/test stats don't influence training

Each feature independently standardized: `(x - mean) / std`

## 7. Model Initialization (Lines 248–257)

```python
model = STGNN(NUM_FEATURES=14, hidden_dim=64).to(device)
edge_index = edge_index.to(device)
x_mean = scaler.x_mean.to(device)
x_std = scaler.x_std.to(device)
```

**STGNN** architecture:
- **Input**: (Batch, SEQ_LEN, NUM_NODES, NUM_FEATURES) = (B, 48, 4, 14)
- **Hidden dimension**: 64 (increased from 32 for better capacity)
- **Output**: (Batch, 3) → [P10, P50, P90] quantiles via quantile loss
- **Mechanism**: Temporal convolution + graph attention/message passing

Device: CUDA → MPS (Apple Silicon) → CPU fallback

## 8. Training Loop (Lines 268–342)

### 8.1 Hyperparameters
```python
optimizer = torch.optim.Adam(lr=5e-4, weight_decay=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(patience=5, factor=0.5)
early_stopping = opt.EarlyStopping(patience=15, min_delta=1e-4)

BATCH_SIZE = 64
EPOCHS = 200  # upper bound
```

- **Learning rate**: 5e-4 (lowered from 1e-3 for stability)
- **Weight decay**: 1e-3 (L2 regularization)
- **Scheduler**: Reduce LR by 0.5× if val loss plateaus for 5 epochs
- **Early stopping**: Stop if val loss doesn't improve by 1e-4 for 15 epochs

### 8.2 Per-Epoch Training
```python
for epoch in range(EPOCHS):
    model.train()
    for batch indices:
        xb, yb = batch
        preds = model(xb, edge_index)
        loss = helper.quantile_loss(preds, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
```

**Quantile loss**: Custom loss that minimizes prediction error for P10, P50, P90 simultaneously. Ensures well-calibrated uncertainty estimates.

**Gradient clipping**: Prevent exploding gradients (norm ≤ 1.0).

### 8.3 Validation & Early Stopping
```python
model.eval()
for val_batch:
    v_loss = helper.quantile_loss(model(xb_v, edge_index), yb_v)
    val_losses_b.append(v_loss.item())

avg_val_loss = np.mean(val_losses_b)
scheduler.step(avg_val_loss)
early_stopping.step(avg_val_loss, model, epoch)
if early_stopping.stop:
    break
```

Checkpoints best model (lowest val loss) and restores before evaluation.

## 9. Evaluation on January 2026 (Lines 348–382)

```python
model.eval()
with torch.no_grad():
    X_test_device = X_test_t.to(device)
    test_preds = model(X_test_device, edge_index).cpu().numpy()  # Shape: (744, 3)

p10 = scaler.inverse_y(test_preds[:, 0])
p50 = scaler.inverse_y(test_preds[:, 1])
p90 = scaler.inverse_y(test_preds[:, 2])
actual = scaler.inverse_y(y_test_t.squeeze().numpy())
```

**Metrics**:
- **MAE**: Mean absolute error between P50 and actual
- **RMSE**: Root mean square error
- **P10–P90 coverage**: Fraction of actual values within prediction interval (target ≥ 80%)
- **Mean/min/max supply**: Forecast statistics

## 10. Scenario Engine (Lines 386–456)

### 10.1 Function Signature
```python
def run_scenario(name, isolate=False, wind_series=None, extra_wind_mw=0):
```

- **isolate**: Remove cross-border edges (Estonia fully island mode)
- **wind_series**: Np.array of counterfactual wind production (48 values per sequence)
- **extra_wind_mw**: Flat MW capacity boost

### 10.2 Isolation Scenario (S2)
```python
if isolate:
    edges = torch.zeros((2, 0), dtype=torch.long)  # Empty edge list
    
    fi_flow_orig = x_mod[:, :, 0, FLOW_FI_IDX].clone()
    lv_flow_orig = x_mod[:, :, 0, FLOW_LV_IDX].clone()
    
    x_mod[:, :, 0, FLOW_FI_IDX] = 0.0
    x_mod[:, :, 1, FLOW_FI_IDX] = 0.0
    x_mod[:, :, 0, FLOW_LV_IDX] = 0.0
    x_mod[:, :, 2, FLOW_LV_IDX] = 0.0
    
    x_mod[:, :, 0, SUPPLY_IDX] += fi_flow_orig + lv_flow_orig
```

**Logic**:
1. Zero out all cross-border flows (EE can't trade)
2. Save original flow values (negative = imports)
3. Add imports back to available energy (convert from flows to supply)
4. Result: EE can only use domestic production

### 10.3 Wind Scenario Injection
```python
if wind_series is not None:
    renew_std = x_std[0, 0, 0, RENEW_IDX].item()
    prod_std = x_std[0, 0, 0, PROD_IDX].item()
    supply_std = x_std[0, 0, 0, SUPPLY_IDX].item()
    wind_mw_std = x_std[0, 0, 0, WIND_MW_IDX].item()
    
    for t in range(len(x_mod)):
        w = wind_series[t : t + SEQ_LEN]
        if len(w) == SEQ_LEN:
            wt = torch.from_numpy(w.astype(np.float32))
            x_mod[t, :, 0, RENEW_IDX] += wt / renew_std
            x_mod[t, :, 0, PROD_IDX] += wt / prod_std
            x_mod[t, :, 0, SUPPLY_IDX] += wt / supply_std
            x_mod[t, :, 0, WIND_MW_IDX] += wt / wind_mw_std
```

**Key points**:
- **wind_series**: Counterfactual wind delta (already has baseline subtracted, e.g., "+323 MW new")
- **For each sequence t**: Take 48 hours of wind data (aligns with input history)
- **Normalize independently**: Each feature divided by its own training std (no ratio scaling)
- **Update 4 features**: RENEW, PROD, SUPPLY, WIND_MW all affected by wind injection
- **Why 4 features?** Wind affects renewable source (RENEW), total production (PROD), available supply (SUPPLY), and is tracked separately (WIND_MW)

### 10.4 Scenario Execution
```python
p50_s1, p10_s1, p90_s1 = run_scenario("S1: Full grid — all connections intact")
p50_s2, p10_s2, p90_s2 = run_scenario("S2: Full isolation", isolate=True)
p50_s3, p10_s3, p90_s3 = run_scenario(
    "S3: Isolated + Scenario A", isolate=True, wind_series=wind_scenA)
p50_s4, p10_s4, p90_s4 = run_scenario(
    "S4: Isolated + Scenario B", isolate=True, wind_series=wind_scenB)
```

## 11. Wind Production Scenarios (Lines 459–473)

```python
wind_scenarios = pd.read_csv("../data/wind_production_scenarios.csv")
baseline = wind_scenarios["wind_mwh_baseline"].values
wind_scenA = (
    wind_scenarios["wind_mwh_scenA"] - wind_scenarios["wind_mwh_baseline"]
).values  # +323 MW new
wind_scenB = (
    wind_scenarios["wind_mwh_scenB"] - wind_scenarios["wind_mwh_baseline"]
).values  # +887 MW new
```

CSV generated by `ursula_wind_counterfactual.ipynb`:
- **Baseline**: January 2026 actual wind (50% below historical average)
- **Scenario A**: Add wind farms (Lääneranna, Pärnu, Aidu) = +323 MW
- **Scenario B**: Add all pipeline farms (5 municipalities) = +887 MW total new capacity

Subtraction ensures we only inject the *additional* wind, not double-count baseline.

## 12. Visualization (Lines 510–595)

### 12.1 Plot 1: Training Curves
```python
axes[0, 0].plot(train_losses, label="Train")
axes[0, 0].plot(val_losses, label="Val")
```

Shows train/val loss over epochs. Indicates overfitting (divergence) or early stopping trigger.

### 12.2 Plot 2: Forecast vs Actual (S1)
```python
axes[0, 1].fill_between(jan_hours, p10, p90, alpha=0.15, label="P10–P90 band")
axes[0, 1].plot(jan_hours, p50, label="P50 forecast")
axes[0, 1].plot(jan_hours, actual, linestyle="--", label="Actual balance")
```

Calibration check: Does actual fall within prediction interval 80%+ of the time?

### 12.3 Plot 3: Scenario Comparison Over Time
```python
axes[1, 0].plot(jan_hours, p50_s1, label="Baseline")
axes[1, 0].plot(jan_hours, p50_s2, label="Isolated (no wind)")
axes[1, 0].plot(jan_hours, p50_s3, label="Scenario A (established plans)")
axes[1, 0].plot(jan_hours, p50_s4, label="Scenario B (pipeline farms)")
axes[1, 0].fill_between(jan_hours, p50_s1, p50_s2, alpha=0.15, label="Import dependency gap")
```

Shows resilience impact of wind investments.

### 12.4 Plot 4: Mean & Worst Supply by Scenario
```python
axes[1, 1].bar(x, mean_supply, color=colors, label="Mean available supply (P50)")
axes[1, 1].scatter(x, min_supply, marker="v", label="Worst hour (P10)")
axes[1, 1].axhline(900, linestyle="--", label="Typical consumption (~900 MW)")
```

Summary statistics: Which scenarios can meet typical 900 MW demand?

## Summary of Data Flow

```
Elering API + Open-Meteo Weather
    ↓
Resample to hourly, merge flows/prices/production/weather
    ↓
Engineer 14 features per node (lag24, calendar, stress signals)
    ↓
Stack into (T, 4 nodes, 14 features)
    ↓
Create 48-hour input / 24-hour target sequences
    ↓
Temporal split: Train (2019–Oct 2025) | Val (Oct–Dec 2025) | Test (Jan 2026)
    ↓
Normalize via PowerScaler (fit on train only)
    ↓
Train ST-GNN with quantile loss, early stopping, gradient clipping
    ↓
Evaluate on unseen Jan 2026 (744 hours)
    ↓
Run 4 scenarios: baseline, isolation, isolation+wind A, isolation+wind B
    ↓
Visualize: training curves, calibration, scenario comparison, summary stats
```

## Key Design Decisions

| Decision | Rationale |
|----------|----------|
| **lag24 feature** | Prevents target leakage while preserving predictive signal |
| **Temporal split** | Simulates real-world forecasting; no future-peeking |
| **Quantile regression** | Risk-aware resilience evaluation (P10 worst-case, P50 mean, P90 best) |
| **4-node graph** | Models actual interconnections; Russia excluded due to poor data |
| **Independent feature normalization** | Correct scaling when injecting counterfactual wind |
| **Baseline wind subtraction** | Prevents double-counting existing wind in scenarios |
| **Early stopping + scheduler** | Avoids overfitting; reduces LR on plateau |